In [1]:
import numpy as np
import h5py
import pickle

Illustris API request function

In [2]:
import requests
import time

baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":"c489f7f5a9e0e61f5060fd16e9fb975e"}

data_dir = './illustris_data/'

def get(path, params=None):
    max_retry = 10

    # make HTTP GET request to path
    for n in range(max_retry):
        try:
            r = requests.get(path, params=params, headers=headers)
            # raise exception if response code is not HTTP SUCCESS (200)
            r.raise_for_status()
            break
        except requests.HTTPError:
            if n < max_retry-1:
                time.sleep(min(2**n, 30))
                continue
            else:
                raise TimeoutError()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = data_dir + r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        f.close()
        return filename # return the filename string

    return r

## Pick primary halo

In [3]:
sim = get(baseUrl+'TNG50-1/')
snaps = get(sim['snapshots'])

In [6]:
# find a MW-like primary halo:

halo_index=0
while True:
    primary_halo = get(snaps[-1]['url']+f'halos/{halo_index}/')
    halo_info = get(primary_halo['meta']['info'])

    halo_mass = 0.7*halo_info['GroupMassType'][1] # get mass of dark matter halo componenet (accounting for hubble constant units)

    if 50 <= halo_mass <= 200:
        print(f"MW-like halo found at index {halo_index}!")
        break
    else:
        halo_index += 1
        continue

MW-like halo found at index 59!


## Calculate groups that would be smoothed together by WDM

In [ ]:
def free_stream_suppress(pos1, pos2):
    """
    Return True if pos11 and pos2 are within the WDM free-streaming distance
    """

    l_fs = 2000 # kpc/h; characteristic free-streaming length for m_wdm=1keV, per [Andrew J. Long and Moira Venegas JCAP06(2025)043]

    dist = np.sum((pos1-pos2)**2)

    if dist <= l_fs**2:
        return True
    else:
        return False

In [ ]:
def wdm_merge(primary_halo):
    # get list of present-day subhalos
    subhalos = get(primary_halo['meta']['url'],{'limit':primary_halo['child_subhalos']['count']})
    subhalo_list = subhalos['child_subhalos']['results']

    merge_groups = []

    failed_list = []

    for i, pointer in enumerate(subhalo_list):
        subhalo = get(pointer['url'])
        try:
            prog_branch = h5py.File(get(subhalo['trees']['sublink_mpb']), 'r')
        except TimeoutError as e:
            print(f"Failed to download progenitor branch for subhalo {pointer['id']}; maximum retries exceeded")
            failed_list.append(pointer)
            continue
            

        SubfindID = prog_branch['SubfindID'][0]
        snap0_pos = prog_branch['SubhaloPos'][-1] # snapshots listed in reverse order here

        closest_subhalo = None
        closest_group = None

        for group in merge_groups:
            for sh in group:
                pos2 = group[sh]

                if not closest_subhalo:
                    closest_subhalo = sh
                    closest_group = group

                elif np.sum((snap0_pos-pos2)**2) < np.sum((snap0_pos-closest_subhalo)**2):
                    closest_subhalo = sh
                    closest_group = group
                
        if not closest_subhalo:
            merge_groups.append({SubfindID: snap0_pos})
            continue

        if free_stream_suppress(snap0_pos, closest_subhalo):
            closest_group[SubfindID] = snap0_pos
        else:
            merge_groups.append({SubfindID: snap0_pos})

        prog_branch.close()

    return merge_groups, failed_list


In [ ]:
for idx in range(91,101): # NOTE: this takes ~6 hours to run per main halo (lots of subhalos!)
    primary_halo = get(snaps[-1]['url']+f'halos/{idx}/')

    merge_groups, failed_list = wdm_merge(primary_halo)

    # save merge data for later use
    import pickle
    with open(f'illustris_data/merges/halo{idx}_merge_groups.pkl', 'wb') as merge_file:
        pickle.dump(merge_groups, merge_file)
    with open(f'illustris_data/merges/halo{idx}_failed_subhalos.pkl', 'wb') as failed_file:
        pickle.dump(failed_list, failed_file)
    print(f"Computed WDM merges for halo {idx}")

Failed to download progenitor branch for subhalo 456341; maximum retries exceeded
Failed to download progenitor branch for subhalo 456345; maximum retries exceeded
Failed to download progenitor branch for subhalo 456380; maximum retries exceeded


In [ ]:
def barycenter_merge(merge_groups):
    
    subhalo_cat = {}
    
    for group in merge_groups:
        if len(group) ==1:
           # get mass of single halo
           subhalo_index = list(group.keys())[0]
           coords = group[subhalo_index]
           subhalo_data = get(snaps[-1]['url']+f'subhalos/{subhalo_index}/')
           mass = subhalo_data['mass']
           subhalo_info = {"pos":coords, "mass":mass}
           subhalo_cat[subhalo_index] = subhalo_info
        else:
            subhalo_indices = list(group.keys())
            subhalo_masses = [get(snaps[-1]['url']+f'subhalos/{i}/')['mass'] for i in subhalo_indices]
            subhalo_coords = [group[i] for i in subhalo_indices]                
            m_tot = np.sum(subhalo_masses)
            weighted_coords = [subhalo_coords[i] * subhalo_masses[i]/m_tot for i in subhalo_indices]
            barycenter_pos = np.sum(weighted_coords)
            merged_halo_info = {"mass":m_tot, "pos":barycenter_pos}
            subhalo_cat[group[0]] = merged_halo_info    

    return subhalo_cat
        

In [ ]:
with open('./illustris_data/merges/halo59_merge_groups.pkl', 'rb') as file:
    merge_groups = pickle.load(file)

    merged_subhalos = barycenter_merge(merge_groups)